1. Import Libraries
2. Load Dataset
3. Data Cleaning
4. Snowball Stemming
5. Lemmatization
6. Train-Test Split
7. Bag of Words
8. TF-IDF
9. Train Model
10. Evaluate
11. Compare Results

In [2]:
import pandas as pd
messages=pd.read_csv('SMSSpamCollection.txt',
                    sep='\t',names=["label","message"])

In [3]:
from nltk.stem.porter import PorterStemmer

ps = PorterStemmer()

In [4]:
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer

english_stopwords = set(stopwords.words("english"))

stemmer = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

In [5]:
import re

def preprocess_with_stemming(text):

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    text = text.lower()

    words = text.split()

    words = [
        stemmer.stem(word)
        for word in words
        if word not in english_stopwords
    ]

    return " ".join(words)

In [6]:
stemmed_corpus = [
    preprocess_with_stemming(text)
    for text in messages["message"]
]

In [7]:
def preprocess_with_lemmatization(text):

    text = re.sub(r'[^a-zA-Z]', ' ', text)

    text = text.lower()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in english_stopwords
    ]

    return " ".join(words)

In [8]:
lemmatized_corpus = [
    preprocess_with_lemmatization(text)
    for text in messages["message"]
]

In [14]:
y = pd.get_dummies(messages["label"])

y = y.iloc[:,0].values

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

In [16]:
def evaluate_model(corpus, vectorizer):

    X_train, X_test, y_train, y_test = train_test_split(
        corpus,
        y,
        test_size=0.20,
        random_state=42
    )

    X_train = vectorizer.fit_transform(X_train)

    X_test = vectorizer.transform(X_test)

    model = MultinomialNB()

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)

    accuracy = accuracy_score(
        y_test,
        predictions
    )

    return accuracy
    

In [17]:
bow_vectorizer = CountVectorizer(max_features=5000)

stem_bow_accuracy = evaluate_model(
    stemmed_corpus,
    bow_vectorizer
)

print(stem_bow_accuracy)

0.9838565022421525


In [18]:
bow_vectorizer = CountVectorizer(max_features=5000)

lemma_bow_accuracy = evaluate_model(
    lemmatized_corpus,
    bow_vectorizer
)

print(lemma_bow_accuracy)

0.9847533632286996


In [19]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

stem_tfidf_accuracy = evaluate_model(
    stemmed_corpus,
    tfidf_vectorizer
)

print(stem_tfidf_accuracy)

0.9730941704035875


In [20]:
tfidf_vectorizer = TfidfVectorizer(max_features=5000)

lemma_tfidf_accuracy = evaluate_model(
    lemmatized_corpus,
    tfidf_vectorizer
)

print(lemma_tfidf_accuracy)

0.9713004484304932


In [21]:
comparison = pd.DataFrame({

    "Preprocessing": [

        "Snowball + BoW",
        "Lemmatization + BoW",
        "Snowball + TF-IDF",
        "Lemmatization + TF-IDF"

    ],

    "Accuracy": [

        stem_bow_accuracy,
        lemma_bow_accuracy,
        stem_tfidf_accuracy,
        lemma_tfidf_accuracy

    ]

})

comparison

,Preprocessing,Accuracy
0,Snowball + BoW,0.983857
1,Lemmatization + BoW,0.984753
2,Snowball + TF-IDF,0.973094
3,Lemmatization + TF-IDF,0.971300
